In [8]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient
# from qdrant_client.models import Filter, FieldCondition, MatchValue



### Instructor

makes sure the output is properly structured
includes retries

-> spefic schema returned and the llm fails
-> instructor retries the same call with the error trace that it gets from the llm invocation so it gives llm additional informaiton where it failed 
-> CONTEXT ENGINEERING

-> top multi language libraryu for structured LLM OUTPUTS
-> will see how instructor falls short 



In [9]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [10]:
prompt =  """you are a helpful assistant
Return an answer to the question
Question: what is your name?"""


response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    reasoning_effort="none",
    messages=[
        {"role": "system", "content": prompt}
    ]
)

print(response.choices[0].message.content)

I’m ChatGPT.


In [11]:
response

ChatCompletion(id='chatcmpl-Dxe1YJWxq7tUs9bM6KMJrLydfBMoA', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m ChatGPT.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1783107576, model='gpt-5.4-nano-2026-03-17', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=8, prompt_tokens=26, total_tokens=34, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

### Add insturctor (Structured outputs)

In [12]:
client = instructor.from_provider("openai/gpt-5.4-nano", mode=instructor.Mode.RESPONSES_TOOLS) # can only add reasoning effort with the responses api
# will return output structured in pydantic data structure

In [ ]:
class Answer(BaseModel):
    answer: str = Field(description="Answer to the question") # the descirption is also injected as context 
    

In [16]:
response = client.create(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
)

In [17]:
response

Answer(answer='I’m ChatGPT, an AI assistant.')

In [18]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=Answer
) 

In [19]:
response


Answer(answer='I’m ChatGPT.')

In [ ]:
raw_response #gives use token usage.... total_tokens is input_tokens + output_tokens

Response(id='resp_0179f04437e5a540006a4810424f04819aa6614a050ad94fea', created_at=1783107650.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT."}', call_id='call_GZJWP3z8FEuTSGUK8xprwZ6F', name='Answer', type='function_call', id='fc_0179f04437e5a540006a481042d45c819aae0971fdb6dda390', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with correct types')], top_p=0.98, background=False, comp

In [21]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="Reasoning for the answer") # should go before the answer because whatever is generated as an answer can only benefit from reasoning if reasoning happens before the answer ^ accuracy of the model
    answer: str = Field(description="Answer to the question")
    

In [22]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning
) 

In [24]:
response

AnswerWithReasoning(reasoning='The user asks for my name. As an AI assistant, I should identify myself succinctly.', answer='I’m ChatGPT, an AI assistant.')

### RAG pipeline 

In [33]:
class RAG_GenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [34]:
qdrant_client = QdrantClient( # when we are running inside of docker network 
    url="http://localhost:6333"
)

def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }
    
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

def generate_answer(prompt):
    # re write this to use instructor
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        response_model=RAG_GenerationResponse,
        reasoning={"effort": "none"}
    )
    return response

def rag_pipeline(question, topk_k=5):
    retrievedContext = retrieve_data(question, k=topk_k)
    processed_context = process_context(retrievedContext)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt) ## llm call w/ prompt
    
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrievedContext["retrieved_context_ids"], ## surface to use for evaluation of precision and recall...
        "retrieved_context": retrievedContext["retrieved_context"],
        "similarity_scores": retrievedContext["similarity_scores"]
    }
    
    return final_answer

In [35]:
output = rag_pipeline("Any USB chargable speaker?")

In [36]:
output

{'data_object': RAG_GenerationResponse(answer='Yes. The Raymate Bluetooth Speakers (B0C996WY16) is a portable wireless speaker with a 3600mAh high-capacity battery and Type-C charging port, giving 1000+ minutes of playtime. It’s also IPX7 waterproof for use indoors and outdoors.'),
 'answer': 'Yes. The Raymate Bluetooth Speakers (B0C996WY16) is a portable wireless speaker with a 3600mAh high-capacity battery and Type-C charging port, giving 1000+ minutes of playtime. It’s also IPX7 waterproof for use indoors and outdoors.',
 'question': 'Any USB chargable speaker?',
 'retrieved_context_ids': ['B0C996WY16',
  'B0C9QZS95R',
  'B0BXC72RLD',
  'B09X9838WY',
  'B0CC4HBS85'],
 'retrieved_context': ['Raymate Bluetooth Speakers, HiFi Stereo Sound with DSP, 30W IPX7 Waterproof Speaker Wireless Bluetooth-V5.0, 1000mins Playtime, Portable Speaker for Home, Outdoor, Party: 🎶HiFi Sound: With proven audio processing DSP chip technology, 30W dual speaker drivers and a more powerful amplifier module, 